<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/4.2-feature-extraction-rav_and_crema.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Alo alo vecinillo

In [6]:
import kagglehub
import os
from google.colab import drive
drive.mount('/content/drive')
# Download latest version desde Kaggle
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")
path_crema = kagglehub.dataset_download("ejlok1/cremad")
FAST_ROOT_DIR = '/content/features_local/ravdess-and-crema-images'
BASE_DIR_RAV = "/kaggle/input/ravdess-emotional-speech-audio"
BASE_DIR_CREMA = "/kaggle/input/cremad"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Using Colab cache for faster access to the 'cremad' dataset.


In [2]:
# Import de librerias necesarias

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import logging
import sys

In [11]:
# Copiamos la carpeta entera de features desde Drive al disco de Colab
#!cp -r /content/drive/MyDrive/ravdess_images_02/ /content/features_local
#os.makedirs(FAST_ROOT_DIR, exist_ok=True)
# Opcional: Si tienes un archivo .zip en Drive, es AÚN MÁS RÁPIDO copiar el .zip y descomprimirlo localmente:
!cp -r /kaggle/input/ravdess-emotional-speech-audio /content/features_local/ravdess-and-crema-images
!cp -r /kaggle/input/cremad /content/features_local/ravdess-and-crema-images
#!unzip -q /content/ravdess_images_02.zip -d /content/features_local


In [12]:
# Unificando convecion de nombres y archivos antes de procesar los clips
os.listdir(FAST_ROOT_DIR)
crema_path = '/content/features_local/ravdess-and-crema-images/cremad'
cremad_list = os.listdir(crema_path)

file_emotion = []
file_path = []

for file in cremad_list:
  # Guardando paths
  file_path.append(crema_path + file)
  # Guardando los clips
  part = file.split('_')
  if part[2] == 'SAD':
    file_emotion.append('sad')
  elif part[2] == 'ANG':
    file_emotion.append('angry')
  elif part[2] == 'DIS':
    file_emotion.append('disgust')
  elif part[2] == 'FEA':
    file_emotion.append('fearful')
  elif part[2] == 'HAP':
    file_emotion.append('happy')
  elif part[2] == 'NEU':
    file_emotion.append('neutral')
  else :
    file_emotion.append('unknown')



['ravdess-emotional-speech-audio', 'cremad']

In [24]:
# Variables de configuración y rutas
#--------------------------------------------------------------------------
SAMPLE_RATE = 22050
MIN_DURATION = 0.5
MAX_DURATION = 3
GENERATE_CSV = True
GENERATE_IMAGES = False # Poner en False para solo obtener DF (Parte CSV)
PAD_MODE = "constant" # Para padding de audios cortos
OUT_DIR_IMAGES = '/content/drive/MyDrive/ravdess_and_crema_images/'
OUT_DIR_CSV = '/content/drive/MyDrive/ravdess_features/features_extended.csv'
BATCH_SIZE_CSV = 60 # Depende cuanto RAM usemos
IMG_RES = (224,224) # Para ResNet
file_path = FAST_ROOT_DIR

#----------------------------

In [33]:
# Extraccion de caracteristicas (De audio a: dataframe(csv) - imagen(png)):
# Celda para genracion de CSV
#----------------------------------------------

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    stream=sys.stdout,
                    force=True
                    )

#Contadores para proceso de los clips

processed_count=0
discarded_count=0
error_count=0

# Funciones para la extraccion:

# Sabemos que en eg. RAVDESS: 03-01-XX-01-01-01-01.wav -> XX= corresponde a la emoción

# Función para agrupar por emoción



def get_emotion_from_filename(filename):
    # Lógica para RAVDESS (ej: 03-01-05-01-01-01-01.wav)
    if len(filename.split('-')) == 7:
        emotion_code = int(filename.split('-')[2])
        # Nota técnica: El código 2 (calm) se mapea directamente a 'neutral'
        rav_map = {1: 'neutral', 2: 'neutral', 3: 'happy', 4: 'sad',
                   5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}
        return rav_map.get(emotion_code, 'unknown')

    # Lógica para CREMA-D (ej: 1001_DFA_ANG_XX.wav)
    elif len(filename.split('_')) >= 3:
        emo_code = filename.split('_')[2].upper()
        # CREMA-D tiene 6 emociones principales
        crema_map = {'NEU': 'neutral', 'HAP': 'happy', 'SAD': 'sad',
                     'ANG': 'angry', 'FEA': 'fearful', 'DIS': 'disgust'}
        return crema_map.get(emo_code, 'unknown')

    return 'unknown'

# Función process_audio
def process_audio(file_path, sr=SAMPLE_RATE, min_dur=MIN_DURATION, max_dur=MAX_DURATION):
    global discarded_count, error_count
    try:
        # Como vimos, es mas conveninete no ser tan agresivos y se procede a usar 40 en top_db
        y, sr = librosa.load(file_path, sr=sr)
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)
        y_trimmed = librosa.util.normalize(y_trimmed)
        duration = len(y_trimmed) / sr
        target_len = int(max_dur * sr)
        y = y_trimmed


        if duration < min_dur:
            logging.info(f"Descartado por insignificante xd {os.path.basename(file_path)}: duración {duration:.2f}s")
            discarded_count += 1
            return None


        #  Escencial en operaciones matriciales

        if len(y_trimmed) < target_len:
            y_trimmed = np.pad(y_trimmed, (0, target_len - len(y_trimmed)), mode=PAD_MODE)

        return y_trimmed, sr
    except Exception as e:
        logging.error(f"Error en {file_path}: {e}")
        discarded_count += 1
        return None


def extract_features(y, sr):
    try:
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20) # Vamos esta vez con 30 coeficientes de Mel (filtros)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048)
        mel_spec_db = librosa.power_to_db(mel_spec)
        rmse = librosa.feature.rms(y=y)      # Solo para CSV
        delta = librosa.feature.delta(mfccs) # Solo para CSV
        delta2 = librosa.feature.delta(mfccs, order=2)  # Solo para CSV

        features = {
            'mfcc_mean': np.mean(mfccs, axis=1), 'mfcc_std': np.std(mfccs, axis=1),
            'delta_mean': np.mean(delta, axis=1), 'delta_std': np.std(delta, axis=1),
            'delta2_mean': np.mean(delta2, axis=1), 'delta2_std': np.std(delta2, axis=1),
            'rmse_mean': np.mean(rmse), 'rmse_std': np.std(rmse),
            'mel_mean': np.mean(mel_spec_db), 'mel_std': np.std(mel_spec_db)
        }
        flat_features = np.concatenate([
            features['mfcc_mean'], features['mfcc_std'], features['delta_mean'], features['delta_std'],
            features['delta2_mean'], features['delta2_std'],
            [features['rmse_mean'], features['rmse_std'],
             features['mel_mean'], features['mel_std']]
        ])
        return flat_features, mfccs, delta, delta2, rmse, mel_spec_db
    except Exception as e:
        logging.error(f"Error features: {e}")
        return None, None, None, None, None, None


# Debug inicial
print(f"Path del dataset: {FAST_ROOT_DIR}")
print(f"Duración rango: {MIN_DURATION}s - {MAX_DURATION}s")
print(f"Generar imágenes: {GENERATE_IMAGES}")


# Main con lotes

files_list = [os.path.join(root, file) for root, dirs, files in os.walk(FAST_ROOT_DIR) for file in files if file.endswith('.wav')]
logging.info(f"Total archivos: {len(files_list)}")

#Bandera para no genrar nuevamente el csv:
if GENERATE_CSV:


    features_list = []
    labels = []
    for i in range(0, len(files_list), BATCH_SIZE_CSV):
        batch_files = files_list[i:i+BATCH_SIZE_CSV]
        logging.info(f"Procesando batch {i//BATCH_SIZE_CSV + 1} / {(len(files_list)//BATCH_SIZE_CSV) + 1}")
        for file_path in batch_files:
            processed = process_audio(file_path)
            if processed:
                y, sr = processed
                feats, mfccs, delta, delta2, rmse, mel_spec_db = extract_features(y, sr)
                if feats is not None:
                    features_list.append(feats)
                    emotion = get_emotion_from_filename(os.path.basename(file_path))
                    labels.append(emotion)
                    processed_count += 1


        # Liberar memoria
        import gc
        gc.collect()
        logging.info(f"Batch {i//BATCH_SIZE_CSV + 1} completado. Procesados acumulados: {processed_count}, Descartados: {discarded_count}, Errores: {error_count}")

    # Estadísticas finales
    logging.info(f"Procesamiento finalizado. Total procesados: {processed_count}, Descartados: {discarded_count}, Errores: {error_count}")
    print(f"Resumen: Procesados {processed_count}, Descartados {discarded_count}, Errores {error_count}")


    # Crear y guardar DF
    columns = ([f'mfcc_mean_{i}' for i in range(13)] + [f'mfcc_std_{i}' for i in range(13)] +
              [f'delta_mean_{i}' for i in range(13)] + [f'delta_std_{i}' for i in range(13)] +
              [f'delta2_mean_{i}' for i in range(13)] + [f'delta2_std_{i}' for i in range(13)] +
              ['rmse_mean', 'rmse_std', 'spec_mean', 'spec_std', 'mel_mean', 'mel_std'])
    df = pd.DataFrame(features_list, columns=columns)
    df['emotion'] = labels
    scaler = StandardScaler()
    df[columns] = scaler.fit_transform(df[columns])
    df.to_csv(OUT_DIR_CSV, index=False)
    logging.info(f"DF guardado en {OUT_DIR_CSV}: {len(df)} filas")

else:
  print("La generacion del archivo CSV está desactivada")

Path del dataset: /content/features_local/ravdess-and-crema-images
Duración rango: 0.5s - 3s
Generar imágenes: False
2026-04-11 05:29:04,626 - INFO - Total archivos: 10322
2026-04-11 05:29:04,628 - INFO - Procesando batch 1 / 173
2026-04-11 05:29:06,776 - INFO - Batch 1 completado. Procesados acumulados: 60, Descartados: 0, Errores: 0
2026-04-11 05:29:06,778 - INFO - Procesando batch 2 / 173
2026-04-11 05:29:09,181 - INFO - Batch 2 completado. Procesados acumulados: 120, Descartados: 0, Errores: 0
2026-04-11 05:29:09,182 - INFO - Procesando batch 3 / 173
2026-04-11 05:29:13,117 - INFO - Batch 3 completado. Procesados acumulados: 180, Descartados: 0, Errores: 0
2026-04-11 05:29:13,119 - INFO - Procesando batch 4 / 173
2026-04-11 05:29:17,518 - INFO - Batch 4 completado. Procesados acumulados: 240, Descartados: 0, Errores: 0
2026-04-11 05:29:17,519 - INFO - Procesando batch 5 / 173
2026-04-11 05:29:19,772 - INFO - Batch 5 completado. Procesados acumulados: 300, Descartados: 0, Errores: 0

ValueError: 84 columns passed, passed data had 124 columns

In [28]:
TARGET_SAMPLES = int(SAMPLE_RATE * MAX_DURATION) # 66,150 muestras exactas

FEATURES_TO_EXTRACT = ['mel_spec', 'mfcc']

# Crear estructura de carpetas base
for feature in FEATURES_TO_EXTRACT:
    os.makedirs(os.path.join(OUT_DIR_IMAGES, feature), exist_ok=True)

# 1. FUNCIÓN UNIFICADA DE ETIQUETAS (RAVDESS + CREMA-D)
def get_unified_emotion(filename):
    # Lógica para RAVDESS (ej: 03-01-05-01-01-01-01.wav)
    if len(filename.split('-')) == 7:
        emotion_code = int(filename.split('-')[2])
        # Nota técnica: El código 2 (calm) se mapea directamente a 'neutral'
        rav_map = {1: 'neutral', 2: 'neutral', 3: 'happy', 4: 'sad',
                   5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised'}
        return rav_map.get(emotion_code, 'unknown')

    # Lógica para CREMA-D (ej: 1001_DFA_ANG_XX.wav)
    elif len(filename.split('_')) >= 3:
        emo_code = filename.split('_')[2].upper()
        # CREMA-D tiene 6 emociones principales
        crema_map = {'NEU': 'neutral', 'HAP': 'happy', 'SAD': 'sad',
                     'ANG': 'angry', 'FEA': 'fearful', 'DIS': 'disgust'}
        return crema_map.get(emo_code, 'unknown')

    return 'unknown'

# 2. FUNCIÓN DE GENERACIÓN DE IMÁGENES
def save_feature_image(data, sr, out_path, feature_type):
    # Configuramos la figura sin ejes ni bordes (solo los datos puros para la CNN)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.axis('off')

    if feature_type == 'mel_spec':
        # Technical Precision: power_to_db es crucial para la percepción logarítmica
        librosa.display.specshow(librosa.power_to_db(data, ref=np.max),
                                 sr=sr, cmap='inferno', ax=ax)
    elif feature_type == 'mfcc':
        librosa.display.specshow(data, sr=sr, cmap='plasma', ax=ax)

    # Guardado optimizado y limpieza estricta de memoria (Troubleshooting RAM)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0, transparent=True)
    fig.clear()
    plt.close(fig)

# 3. BUCLE PRINCIPAL DE PROCESAMIENTO
files_list = [os.path.join(root, f) for root, dirs, filenames in os.walk(FAST_ROOT_DIR) for f in filenames if f.endswith('.wav')]
print(f"Iniciando procesamiento de {len(files_list)} archivos...")

processed = 0

for i, file_path in enumerate(files_list):
    filename = os.path.basename(file_path)
    emotion = get_unified_emotion(filename)

    if emotion == 'unknown':
        continue

    try:
        # Cargar audio
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)

        # Eliminar silencios iniciales/finales
        y_trimmed, _ = librosa.effects.trim(y, top_db=45)

        # Estandarización de longitud (Padding/Truncating a 3 segundos exactos)
        y_fixed = librosa.util.fix_length(y_trimmed, size=TARGET_SAMPLES)

        # Extracción y guardado para cada feature
        for feature in FEATURES_TO_EXTRACT:
            # Crear subcarpeta de emoción si no existe
            emotion_dir = os.path.join(OUT_DIR_IMAGES, feature, emotion)
            os.makedirs(emotion_dir, exist_ok=True)

            # Definir ruta de salida (usamos prefijos r_ y c_ para evitar colisiones de nombres si las hubiera)
            prefix = "r_" if "-" in filename else "c_"
            out_name = f"{prefix}{filename.replace('.wav', '.png')}"
            out_path = os.path.join(emotion_dir, out_name)

            # Si la imagen ya existe, nos la saltamos (permite reanudar si se corta)
            if os.path.exists(out_path): continue

            # Cálculos de señales
            if feature == 'mel_spec':
                data = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128, fmax=8000)
            elif feature == 'mfcc':
                data = librosa.feature.mfcc(y=y_fixed, sr=sr, n_mfcc=20)

            save_feature_image(data, sr, out_path, feature)

        processed += 1

        # Feedback visual y recolección de basura cada 500 archivos
        if processed % 500 == 0:
            print(f"Procesados {processed}/{len(files_list)} audios...")
            gc.collect()

    except Exception as e:
        print(f"Error procesando {filename}: {e}")

print("¡Extracción de imágenes completada exitosamente!")

Iniciando procesamiento de 10322 archivos...
Procesados 500/10322 audios...
Procesados 1000/10322 audios...
Procesados 1500/10322 audios...
Procesados 2000/10322 audios...
Procesados 2500/10322 audios...
Procesados 3000/10322 audios...
Procesados 3500/10322 audios...
Procesados 4000/10322 audios...
Procesados 4500/10322 audios...
Procesados 5000/10322 audios...
Procesados 5500/10322 audios...
Procesados 6000/10322 audios...
Procesados 6500/10322 audios...
Procesados 7000/10322 audios...
Procesados 7500/10322 audios...
Procesados 8000/10322 audios...
Procesados 8500/10322 audios...
Procesados 9000/10322 audios...
Procesados 9500/10322 audios...
Procesados 10000/10322 audios...
¡Extracción de imágenes completada exitosamente!
